In [3]:
import requests
import pandas as pd
from datetime import datetime, timezone
import zipfile
from pathlib import Path


In [ ]:
import requests
import pandas as pd
from datetime import datetime, timezone

# Futures (USD-M perpetual) base URL — no API key required
FBASE  = "https://fapi.binance.com"
SYMBOL = "BTCUSDT"

# --- 1. Current mark / index / last price ---
premium = requests.get(f"{FBASE}/fapi/v1/premiumIndex", params={"symbol": SYMBOL}).json()
print(f"Last price:   {premium['lastFundingRate']}  (funding rate)")
print(f"Mark price:   {premium['markPrice']}")
print(f"Index price:  {premium['indexPrice']}")
print(f"Next funding: {datetime.fromtimestamp(premium['nextFundingTime']/1000, tz=timezone.utc)}")

# --- 2. 24h stats ---
stats = requests.get(f"{FBASE}/fapi/v1/ticker/24hr", params={"symbol": SYMBOL}).json()
stats_df = pd.DataFrame([{
    "symbol":        stats["symbol"],
    "last_price":    float(stats["lastPrice"]),
    "open":          float(stats["openPrice"]),
    "high":          float(stats["highPrice"]),
    "low":           float(stats["lowPrice"]),
    "volume_btc":    float(stats["volume"]),
    "volume_usdt":   float(stats["quoteVolume"]),
    "price_change%": float(stats["priceChangePercent"]),
    "trades":        stats["count"],
    "timestamp":     datetime.fromtimestamp(stats["closeTime"] / 1000, tz=timezone.utc),
}])
print("\n24h Stats (perp):")
display(stats_df.T)

# --- 3. Order book snapshot (top 20 levels) ---
depth = requests.get(f"{FBASE}/fapi/v1/depth", params={"symbol": SYMBOL, "limit": 20}).json()

bids = pd.DataFrame(depth["bids"], columns=["price", "qty"], dtype=float)
asks = pd.DataFrame(depth["asks"], columns=["price", "qty"], dtype=float)
bids["side"] = "bid"
asks["side"] = "ask"
book_perp = pd.concat([bids, asks], ignore_index=True)

print(f"\nOrder book snapshot  (last update id: {depth['lastUpdateId']})")
print("\nTop 5 bids:")
display(bids.head())
print("\nTop 5 asks:")
display(asks.head())

# --- 4. Recent trades (last 20) ---
trades_raw = requests.get(f"{FBASE}/fapi/v1/trades", params={"symbol": SYMBOL, "limit": 20}).json()
trades_df = pd.DataFrame(trades_raw)[["id", "price", "qty", "time", "isBuyerMaker"]]
trades_df[["price", "qty"]] = trades_df[["price", "qty"]].astype(float)
trades_df["time"] = pd.to_datetime(trades_df["time"], unit="ms", utc=True)
trades_df.rename(columns={"isBuyerMaker": "sell"}, inplace=True)

print("\nRecent trades (perp):")
display(trades_df)

# --- 5. Open interest ---
oi = requests.get(f"{FBASE}/fapi/v1/openInterest", params={"symbol": SYMBOL}).json()
print(f"\nOpen interest: {float(oi['openInterest']):.2f} BTC  "
      f"(as of {datetime.fromtimestamp(oi['time']/1000, tz=timezone.utc)})")

Last price:   -0.00009116  (funding rate)
Mark price:   80943.90000000
Index price:  80962.70847826
Next funding: 2026-05-06 08:00:00+00:00

24h Stats (perp):


,0
symbol,BTCUSDT
last_price,80933.4
open,80050.0
high,81745.4
low,80031.5
volume_btc,174388.349
volume_usdt,14143039737.059999
price_change%,1.104
trades,3584354
timestamp,2026-05-06 00:51:34.515000+00:00



Order book snapshot  (last update id: 10480700182857)

Top 5 bids:


,price,qty,side
0,80943.9,6.386,bid
1,80943.8,0.157,bid
2,80943.7,0.002,bid
3,80943.6,0.002,bid
4,80943.5,0.006,bid



Top 5 asks:


,price,qty,side
0,80944.0,5.217,ask
1,80944.1,0.015,ask
2,80944.2,0.001,ask
3,80944.4,0.004,ask
4,80944.5,0.002,ask



Recent trades (perp):


,id,price,qty,time,sell
0,7633412579,80944.0,0.049,2026-05-06 00:51:41.354000+00:00,False
1,7633412580,80944.0,0.322,2026-05-06 00:51:41.354000+00:00,False
2,7633412581,80944.0,0.002,2026-05-06 00:51:41.354000+00:00,False
3,7633412582,80944.0,0.001,2026-05-06 00:51:41.354000+00:00,False
4,7633412583,80944.0,0.263,2026-05-06 00:51:41.354000+00:00,False
5,7633412584,80944.0,0.029,2026-05-06 00:51:41.354000+00:00,False
6,7633412585,80944.0,0.065,2026-05-06 00:51:41.354000+00:00,False
7,7633412586,80944.0,0.282,2026-05-06 00:51:41.432000+00:00,False
8,7633412587,80944.0,0.520,2026-05-06 00:51:41.432000+00:00,False
9,7633412588,80944.0,0.006,2026-05-06 00:51:41.432000+00:00,False



Open interest: 113746.86 BTC  (as of 2026-05-06 00:51:36.479000+00:00)


In [3]:
liquidation_snapshot_path = '/Users/vadanantoniu/Documents/research/MFin/crypto/data/BTCUSD_PERP-liquidationSnapshot-2024-10-14.csv'
liq_snap_df = pd.read_csv(liquidation_snapshot_path).drop_duplicates().reset_index(drop=True)
liq_snap_df.insert(1, "datetime", pd.to_datetime(liq_snap_df["time"], unit="ms", utc=True))
display(liq_snap_df.head(5))
display(liq_snap_df.tail(5))

,time,datetime,side,order_type,time_in_force,original_quantity,price,average_price,order_status,last_fill_quantity,accumulated_fill_quantity
0,1728866962145,2024-10-14 00:49:22.145000+00:00,SELL,LIMIT,IOC,1,62246.2,62472.0,FILLED,1,1
1,1728867043085,2024-10-14 00:50:43.085000+00:00,SELL,LIMIT,IOC,9,62196.9,62436.4,FILLED,9,9
2,1728867059056,2024-10-14 00:50:59.056000+00:00,SELL,LIMIT,IOC,248,62187.2,62420.9,FILLED,34,248
3,1728875837112,2024-10-14 03:17:17.112000+00:00,BUY,LIMIT,IOC,7,63060.7,62817.3,FILLED,7,7
4,1728875847058,2024-10-14 03:17:27.058000+00:00,BUY,LIMIT,IOC,1,63071.5,62821.4,FILLED,1,1


,time,datetime,side,order_type,time_in_force,original_quantity,price,average_price,order_status,last_fill_quantity,accumulated_fill_quantity
45,1728878507116,2024-10-14 04:01:47.116000+00:00,BUY,LIMIT,IOC,5,64711.7,64473.8,FILLED,5,5
46,1728878529115,2024-10-14 04:02:09.115000+00:00,BUY,LIMIT,IOC,1,64731.9,64500.0,FILLED,1,1
47,1728882332076,2024-10-14 05:05:32.076000+00:00,SELL,LIMIT,IOC,44,63819.5,64070.1,FILLED,44,44
48,1728882421093,2024-10-14 05:07:01.093000+00:00,SELL,LIMIT,IOC,12,63739.4,63963.5,FILLED,12,12
49,1728883631079,2024-10-14 05:27:11.079000+00:00,SELL,LIMIT,IOC,1,63583.3,63889.9,FILLED,1,1


In [ ]:
LIQ_DIR = Path("/Users/vadanantoniu/Documents/research/MFin/crypto/data/BTCUSD_PERP-liquidationSnapshot")

# Unzip all archives in-place (skip if CSV already extracted)
for zp in sorted(LIQ_DIR.glob("*.zip")):
    csv_name = zp.with_suffix(".csv")
    if not csv_name.exists():
        with zipfile.ZipFile(zp) as z:
            z.extractall(LIQ_DIR)

# Read and merge all CSVs
csv_files = sorted(LIQ_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files")

liq_all = (
    pd.concat(
        (pd.read_csv(f) for f in csv_files),
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values("time")
    .reset_index(drop=True)
)

liq_all.insert(1, "datetime", pd.to_datetime(liq_all["time"], unit="ms", utc=True))

print(f"Total rows: {len(liq_all):,}  |  Date range: {liq_all['datetime'].iloc[0]} → {liq_all['datetime'].iloc[-1]}")

Found 472 CSV files
Total rows: 53,398  |  Date range: 2023-06-25 01:27:51.926000+00:00 → 2024-10-14 05:27:11.079000+00:00


In [5]:
print("\n--- head(10) ---")
display(liq_all.head(10))
print("\n--- tail(10) ---")
display(liq_all.tail(10))


--- head(10) ---


,time,datetime,side,order_type,time_in_force,original_quantity,price,average_price,order_status,last_fill_quantity,accumulated_fill_quantity
0,1687656471926,2023-06-25 01:27:51.926000+00:00,BUY,LIMIT,IOC,7,30741.3,30631.6,FILLED,6,7
1,1687656473356,2023-06-25 01:27:53.356000+00:00,BUY,LIMIT,IOC,43,30756.6,30631.6,FILLED,1,43
2,1687659729506,2023-06-25 02:22:09.506000+00:00,BUY,LIMIT,IOC,10,30784.9,30664.4,FILLED,10,10
3,1687660479390,2023-06-25 02:34:39.390000+00:00,BUY,LIMIT,IOC,2,30790.7,30672.9,FILLED,2,2
4,1687662893225,2023-06-25 03:14:53.225000+00:00,BUY,LIMIT,IOC,2,30825.5,30704.8,FILLED,2,2
5,1687663686420,2023-06-25 03:28:06.420000+00:00,BUY,LIMIT,IOC,3,30832.2,30711.0,FILLED,3,3
6,1687663697070,2023-06-25 03:28:17.070000+00:00,BUY,LIMIT,IOC,148,30835.9,30728.6,FILLED,148,148
7,1687665095016,2023-06-25 03:51:35.016000+00:00,BUY,LIMIT,IOC,3,30877.5,30755.0,FILLED,3,3
8,1687665128467,2023-06-25 03:52:08.467000+00:00,BUY,LIMIT,IOC,19,30891.2,30770.0,FILLED,19,19
9,1687665746391,2023-06-25 04:02:26.391000+00:00,BUY,LIMIT,IOC,2,30910.9,30788.3,FILLED,2,2



--- tail(10) ---


,time,datetime,side,order_type,time_in_force,original_quantity,price,average_price,order_status,last_fill_quantity,accumulated_fill_quantity
53388,1728878189054,2024-10-14 03:56:29.054000+00:00,BUY,LIMIT,IOC,163,64493.6,64300.1,FILLED,19,163
53389,1728878412132,2024-10-14 04:00:12.132000+00:00,BUY,LIMIT,IOC,36,64600.6,64376.1,FILLED,36,36
53390,1728878489026,2024-10-14 04:01:29.026000+00:00,BUY,LIMIT,IOC,2,64669.7,64434.9,FILLED,2,2
53391,1728878492137,2024-10-14 04:01:32.137000+00:00,BUY,LIMIT,IOC,270,64683.0,64462.7,FILLED,270,270
53392,1728878494079,2024-10-14 04:01:34.079000+00:00,BUY,LIMIT,IOC,47,64696.4,64469.2,FILLED,19,47
53393,1728878507116,2024-10-14 04:01:47.116000+00:00,BUY,LIMIT,IOC,5,64711.7,64473.8,FILLED,5,5
53394,1728878529115,2024-10-14 04:02:09.115000+00:00,BUY,LIMIT,IOC,1,64731.9,64500.0,FILLED,1,1
53395,1728882332076,2024-10-14 05:05:32.076000+00:00,SELL,LIMIT,IOC,44,63819.5,64070.1,FILLED,44,44
53396,1728882421093,2024-10-14 05:07:01.093000+00:00,SELL,LIMIT,IOC,12,63739.4,63963.5,FILLED,12,12
53397,1728883631079,2024-10-14 05:27:11.079000+00:00,SELL,LIMIT,IOC,1,63583.3,63889.9,FILLED,1,1


In [6]:
out_path = LIQ_DIR / "liquidationSnapshotsAll.csv"
liq_all.to_csv(out_path, index=False)
print(f"Saved {len(liq_all):,} rows → {out_path}")

Saved 53,398 rows → /Users/vadanantoniu/Documents/research/MFin/crypto/data/BTCUSD_PERP-liquidationSnapshot/liquidationSnapshotsAll.csv


In [7]:
liq_all[liq_all['side'] == 'SELL']

,time,datetime,side,order_type,time_in_force,original_quantity,price,average_price,order_status,last_fill_quantity,accumulated_fill_quantity
24,1687668800357,2023-06-25 04:53:20.357000+00:00,SELL,LIMIT,IOC,12,30754.5,30880.0,FILLED,12,12
25,1687670378099,2023-06-25 05:19:38.099000+00:00,SELL,LIMIT,IOC,240,30725.5,30844.5,FILLED,2,240
26,1687670380489,2023-06-25 05:19:40.489000+00:00,SELL,LIMIT,IOC,5,30720.3,30840.0,FILLED,5,5
27,1687672990202,2023-06-25 06:03:10.202000+00:00,SELL,LIMIT,IOC,12,30696.6,30808.3,FILLED,12,12
28,1687673308098,2023-06-25 06:08:28.098000+00:00,SELL,LIMIT,IOC,64,30653.6,30775.1,FILLED,64,64
...,...,...,...,...,...,...,...,...,...,...,...
53349,1728867043085,2024-10-14 00:50:43.085000+00:00,SELL,LIMIT,IOC,9,62196.9,62436.4,FILLED,9,9
53350,1728867059056,2024-10-14 00:50:59.056000+00:00,SELL,LIMIT,IOC,248,62187.2,62420.9,FILLED,34,248
53395,1728882332076,2024-10-14 05:05:32.076000+00:00,SELL,LIMIT,IOC,44,63819.5,64070.1,FILLED,44,44
53396,1728882421093,2024-10-14 05:07:01.093000+00:00,SELL,LIMIT,IOC,12,63739.4,63963.5,FILLED,12,12


In [4]:
data_path = Path("/Users/vadanantoniu/Documents/research/MFin/crypto/data/samples")
book_ticker_path = data_path / 'BTCUSD_PERP-bookTicker-2024-10-14.csv'
book_ticker = pd.read_csv(book_ticker_path)
book_ticker.insert(
    book_ticker.columns.get_loc("event_time") + 1,
    "datetime",
    pd.to_datetime(book_ticker["event_time"], unit="ms", utc=True),
)

In [5]:
book_ticker.head()

,update_id,best_bid_price,best_bid_qty,best_ask_price,best_ask_qty,transaction_time,event_time,datetime
0,1049536582602,62822.4,1602.0,62822.5,2175.0,1728864000278,1728864000485,2024-10-14 00:00:00.485000+00:00
1,1049536582612,62822.4,1613.0,62822.5,2175.0,1728864000278,1728864000485,2024-10-14 00:00:00.485000+00:00
2,1049536582799,62822.4,2512.0,62822.5,2175.0,1728864000279,1728864000485,2024-10-14 00:00:00.485000+00:00
3,1049536582846,62822.4,2516.0,62822.5,2175.0,1728864000279,1728864000486,2024-10-14 00:00:00.486000+00:00
4,1049536583150,62822.4,2516.0,62822.5,2159.0,1728864000281,1728864000494,2024-10-14 00:00:00.494000+00:00


In [ ]:
book_depth_path = data_path / 'BTCUSD_PERP-bookDepth-2024-07-01.csv'
book_depth = pd.read_csv(book_depth_path)

,timestamp,percentage,depth,notional
0,2024-07-01 00:00:00,-5,1255720.0,1898.0
1,2024-07-01 00:00:00,-4,1187993.0,1803.0
2,2024-07-01 00:00:00,-3,1121330.0,1709.0
3,2024-07-01 00:00:00,-2,976457.0,1494.0
4,2024-07-01 00:00:00,-1,519974.0,787.0
...,...,...,...,...
28795,2024-07-01 23:59:30,1,230137.0,336.0
28796,2024-07-01 23:59:30,2,879300.0,1334.0
28797,2024-07-01 23:59:30,3,1164137.0,1761.0
28798,2024-07-01 23:59:30,4,1255390.0,1886.0


In [11]:
book_depth.head(30)

,timestamp,percentage,depth,notional
0,2024-07-01 00:00:00,-5,1255720.0,1898.0
1,2024-07-01 00:00:00,-4,1187993.0,1803.0
2,2024-07-01 00:00:00,-3,1121330.0,1709.0
3,2024-07-01 00:00:00,-2,976457.0,1494.0
4,2024-07-01 00:00:00,-1,519974.0,787.0
5,2024-07-01 00:00:00,1,255718.0,359.0
6,2024-07-01 00:00:00,2,905943.0,1359.0
7,2024-07-01 00:00:00,3,1191912.0,1786.0
8,2024-07-01 00:00:00,4,1309342.0,1953.0
9,2024-07-01 00:00:00,5,1344669.0,1995.0


In [6]:
trades_path = data_path / 'BTCUSD_PERP-trades-2024-10-14.csv'
trades = pd.read_csv(trades_path)
trades.insert(
    trades.columns.get_loc("time") + 1,
    "datetime",
    pd.to_datetime(trades["time"], unit="ms", utc=True),
)

In [7]:
trades.head(5)

,id,price,qty,base_qty,time,datetime,is_buyer_maker
0,872221446,62822.5,39.0,0.062080,1728864001101,2024-10-14 00:00:01.101000+00:00,False
1,872221447,62822.5,11.0,0.017510,1728864001101,2024-10-14 00:00:01.101000+00:00,False
2,872221448,62822.5,1.0,0.001592,1728864001101,2024-10-14 00:00:01.101000+00:00,False
3,872221449,62822.5,38.0,0.060488,1728864001101,2024-10-14 00:00:01.101000+00:00,False
4,872221450,62822.5,39.0,0.062080,1728864001101,2024-10-14 00:00:01.101000+00:00,False


In [15]:
metrics_path = data_path / 'BTCUSD_PERP-metrics-2024-10-14.csv'
metrics = pd.read_csv(metrics_path)

In [16]:
metrics.head(5)

,create_time,symbol,sum_open_interest,sum_open_interest_value,count_toptrader_long_short_ratio,sum_toptrader_long_short_ratio,count_long_short_ratio,sum_taker_long_short_vol_ratio
0,2024-10-14 00:00:00,BTCUSD_PERP,14127335.0,22486.556981,NaN,NaN,NaN,0.676704
1,2024-10-14 00:05:00,BTCUSD_PERP,14128653.0,22507.033841,NaN,NaN,NaN,0.434858
2,2024-10-14 00:10:00,BTCUSD_PERP,14121701.0,22529.224638,NaN,NaN,NaN,0.416428
3,2024-10-14 00:15:00,BTCUSD_PERP,14123248.0,22541.421997,NaN,NaN,NaN,1.423853
4,2024-10-14 00:20:00,BTCUSD_PERP,14114284.0,22552.543781,NaN,NaN,NaN,0.566535


In [21]:
agg_trades_path = data_path / 'BTCUSD_PERP-aggTrades-2024-10-14.csv'
agg_trades = pd.read_csv(agg_trades_path)
agg_trades.insert(
    agg_trades.columns.get_loc("transact_time") + 1,
    "datetime",
    pd.to_datetime(agg_trades["transact_time"], unit="ms", utc=True),
)

In [22]:
agg_trades.head(5)

,agg_trade_id,price,quantity,first_trade_id,last_trade_id,transact_time,datetime,is_buyer_maker
0,358953464,62822.5,564.0,872221446,872221465,1728864001101,2024-10-14 00:00:01.101000+00:00,False
1,358953465,62826.7,25.0,872221466,872221468,1728864001183,2024-10-14 00:00:01.183000+00:00,False
2,358953466,62828.5,201.0,872221469,872221476,1728864001205,2024-10-14 00:00:01.205000+00:00,False
3,358953467,62829.2,25.0,872221477,872221477,1728864001208,2024-10-14 00:00:01.208000+00:00,False
4,358953468,62829.3,10.0,872221478,872221478,1728864001208,2024-10-14 00:00:01.208000+00:00,False
